In [ ]:
# =========================================================
# Install libraries
# =========================================================
!pip install openpyxl scipy statsmodels -q

# =========================================================
# Imports
# =========================================================
import pandas as pd
import numpy as np

from scipy import stats
from statsmodels.stats.multitest import multipletests

from google.colab import files

# =========================================================
# Upload Excel file
# =========================================================
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

# =========================================================
# Read ALL sheets
# =========================================================
excel_file = pd.ExcelFile(file_name)

print("Sheets detected:")
print(excel_file.sheet_names)

# =========================================================
# Function to infer chance level from sheet name
# =========================================================
def infer_chance_level(sheet_name):

    name = sheet_name.lower()

    # E1 -> 14-class
    if "e1" in name or "14" in name:
        return 100 / 14, 14

    # E2 -> words-only (9 classes)
    elif "e2" in name or "word" in name:
        return 100 / 9, 9

    # E3 -> vowels-only (5 classes)
    elif "e3" in name or "vowel" in name:
        return 100 / 5, 5

    # E4 -> binary
    elif "e4" in name:
        return 50.0, 2

    # E5 -> binary
    elif "e5" in name:
        return 50.0, 2

    else:
        return None, None



Saving resultados10corridas.xlsx to resultados10corridas (1).xlsx
Sheets detected:
['E1', 'E2', 'E3', 'E4', 'E5']


In [ ]:
# =========================================================
# Process each sheet
# =========================================================
all_results = []

for sheet in excel_file.sheet_names:

    print("\n=================================================")
    print("Processing:", sheet)

    # -----------------------------------------------------
    # Read sheet
    # -----------------------------------------------------
    df = pd.read_excel(file_name, sheet_name=sheet)

    first_col = str(df.columns[0]).lower()

    if first_col in ["iteración", "iteracion", "iteration"]:
        data = df.iloc[:, 1:]
    else:
        data = df.copy()

    data = data.apply(pd.to_numeric, errors="coerce")

    # -----------------------------------------------------
    # Infer chance level
    # -----------------------------------------------------
    chance_level, n_classes = infer_chance_level(sheet)

    if chance_level is None:
        print("Could not infer chance level.")
        continue

    # -----------------------------------------------------
    # Subject mean accuracies
    # -----------------------------------------------------
    subject_means = data.mean(axis=0).values

    overall_mean = np.mean(subject_means)
    overall_std = np.std(subject_means, ddof=1)

    # -----------------------------------------------------
    # Wilcoxon signed-rank test
    # H1: Accuracy > Chance level
    # -----------------------------------------------------
    w_stat, w_p = stats.wilcoxon(
        subject_means - chance_level,
        alternative="greater"
    )

    print(f"Detected classes : {n_classes}")
    print(f"Chance level     : {chance_level:.4f}")
    print(f"Mean accuracy    : {overall_mean:.4f}")
    print(f"Wilcoxon W       : {w_stat:.1f}")
    print(f"Wilcoxon p-value : {w_p:.6f}")

    # -----------------------------------------------------
    # Store results
    # -----------------------------------------------------
    all_results.append({

        "Experiment": sheet,
        "Classes": n_classes,
        "Chance_Level": chance_level,

        "Mean_Accuracy": overall_mean,
        "Std": overall_std,

        "Wilcoxon_stat": w_stat,
        "Wilcoxon_p": w_p,

        "Above_Chance": w_p < 0.05

    })

# =========================================================
# Final table
# =========================================================
results_df = pd.DataFrame(all_results)

display(results_df)

# =========================================================
# Save
# =========================================================
results_df.to_excel(
    "Wilcoxon_Above_Chance_Analysis.xlsx",
    index=False
)

print("\nResults saved:")
print("Wilcoxon_Above_Chance_Analysis.xlsx")


Processing: E1
Detected classes : 14
Chance level     : 7.1429
Mean accuracy    : 14.5822
Wilcoxon W       : 136.0
Wilcoxon p-value : 0.000015

Processing: E2
Detected classes : 9
Chance level     : 11.1111
Mean accuracy    : 20.8086
Wilcoxon W       : 136.0
Wilcoxon p-value : 0.000015

Processing: E3
Detected classes : 5
Chance level     : 20.0000
Mean accuracy    : 30.7863
Wilcoxon W       : 136.0
Wilcoxon p-value : 0.000015

Processing: E4
Detected classes : 2
Chance level     : 50.0000
Mean accuracy    : 63.5296
Wilcoxon W       : 136.0
Wilcoxon p-value : 0.000015

Processing: E5
Detected classes : 2
Chance level     : 50.0000
Mean accuracy    : 74.6076
Wilcoxon W       : 136.0
Wilcoxon p-value : 0.000015


,Experiment,Classes,Chance_Level,Mean_Accuracy,Std,Wilcoxon_stat,Wilcoxon_p,Above_Chance
0,E1,14,7.142857,14.582164,4.560868,136.0,0.000015,True
1,E2,9,11.111111,20.808578,3.216253,136.0,0.000015,True
2,E3,5,20.000000,30.786325,4.918252,136.0,0.000015,True
3,E4,2,50.000000,63.529581,3.847240,136.0,0.000015,True
4,E5,2,50.000000,74.607643,7.346383,136.0,0.000015,True



Results saved:
Wilcoxon_Above_Chance_Analysis.xlsx
